# Phase 4: Source-Specific Variable Analysis

This notebook presents Phase 4 findings: source attribution for staging variables (ETE, invasion, margins, molecular markers), concordance analysis, and H1/H2 sensitivity results with source-aware covariates.

_Date: 2026-03-12_

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import duckdb, toml
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

secrets = toml.load('../.streamlit/secrets.toml')
token = secrets['MOTHERDUCK_TOKEN']
con = duckdb.connect(f'md:thyroid_research_2026?motherduck_token={token}')
print('Connected to MotherDuck')

## 1. Variable Inventory Summary

In [ ]:
import json
with open('../notes_extraction/variable_inventory_phase4.json') as f:
    inventory = json.load(f)

inv_df = pd.DataFrame([{
    'Variable': v['label'],
    'Priority': v['phase4_priority'],
    'Clinical Impact': v['clinical_impact'],
    'Fill Rate %': v.get('fill_rate_pct', 'N/A'),
    'Precision': f"{v['current_precision_estimate']:.0%}",
    'Source Split': 'YES' if v['requires_source_split'] else '-'
} for v in inventory])

inv_df.sort_values('Priority').style.background_gradient(subset=['Clinical Impact'], cmap='YlOrRd')

## 2. ETE Grade Distribution

In [ ]:
ete_dist = con.execute("""
    SELECT ete_grade, COUNT(*) as n, ROUND(100.0*COUNT(*)/3879.0, 1) as pct
    FROM extracted_ete_refined_v1
    GROUP BY 1 ORDER BY 2 DESC
""").fetchdf()
print('ETE grade distribution (N=3,879 patients with ETE data):')
print(ete_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#c0392b', '#e67e22', '#f39c12', '#95a5a6']
bars = ax.bar(ete_dist['ete_grade'].fillna('null'), ete_dist['n'], color=colors[:len(ete_dist)])
ax.set_xlabel('ETE Grade')
ax.set_ylabel('Number of Patients')
ax.set_title('ETE Grade Distribution (Structured Path Report, N=3,879)')
for bar, n, pct in zip(bars, ete_dist['n'], ete_dist['pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f'{n}\n({pct}%)', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../exports/publication_figures_300dpi/fig_ete_grade_dist.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Source Contribution Heatmap

In [ ]:
import matplotlib.image as mpimg
img = mpimg.imread('../exports/publication_figures_300dpi/fig_source_contribution_heatmap.png')
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
ax.set_title('Source Contribution Heatmap (Phase 4)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Patient Refined Staging Flags Distribution

In [ ]:
staging = con.execute("""
    SELECT
        'ETE path confirmed' as variable,
        SUM(CASE WHEN ete_path_confirmed THEN 1 ELSE 0 END) as n_positive,
        COUNT(*) as total,
        ROUND(100.0*SUM(CASE WHEN ete_path_confirmed THEN 1 ELSE 0 END)/COUNT(*), 1) as pct
    FROM patient_refined_staging_flags_v3
    UNION ALL
    SELECT 'ETE gross', SUM(CASE WHEN ete_grade='gross' THEN 1 ELSE 0 END), COUNT(*),
        ROUND(100.0*SUM(CASE WHEN ete_grade='gross' THEN 1 ELSE 0 END)/COUNT(*), 1)
    FROM patient_refined_staging_flags_v3
    UNION ALL
    SELECT 'Positive margin', SUM(CASE WHEN margin_status_refined='positive' THEN 1 ELSE 0 END), COUNT(*),
        ROUND(100.0*SUM(CASE WHEN margin_status_refined='positive' THEN 1 ELSE 0 END)/COUNT(*), 1)
    FROM patient_refined_staging_flags_v3
    UNION ALL
    SELECT 'Vascular invasion', SUM(CASE WHEN vascular_invasion_refined IS NOT NULL AND vascular_invasion_refined!='indeterminate' THEN 1 ELSE 0 END), COUNT(*),
        ROUND(100.0*SUM(CASE WHEN vascular_invasion_refined IS NOT NULL AND vascular_invasion_refined!='indeterminate' THEN 1 ELSE 0 END)/COUNT(*), 1)
    FROM patient_refined_staging_flags_v3
    UNION ALL
    SELECT 'LVI', SUM(CASE WHEN lvi_refined IN ('present','extensive','focal') THEN 1 ELSE 0 END), COUNT(*),
        ROUND(100.0*SUM(CASE WHEN lvi_refined IN ('present','extensive','focal') THEN 1 ELSE 0 END)/COUNT(*), 1)
    FROM patient_refined_staging_flags_v3
    UNION ALL
    SELECT 'Perineural invasion', SUM(CASE WHEN perineural_invasion_refined='present' THEN 1 ELSE 0 END), COUNT(*),
        ROUND(100.0*SUM(CASE WHEN perineural_invasion_refined='present' THEN 1 ELSE 0 END)/COUNT(*), 1)
    FROM patient_refined_staging_flags_v3
    UNION ALL
    SELECT 'BRAF positive', SUM(CASE WHEN braf_positive_refined THEN 1 ELSE 0 END), COUNT(*),
        ROUND(100.0*SUM(CASE WHEN braf_positive_refined THEN 1 ELSE 0 END)/COUNT(*), 1)
    FROM patient_refined_staging_flags_v3
""").fetchdf()
print('Staging flags (N=10,871 patients):')
staging

## 5. H1 Phase 4 ETE Sensitivity Analysis

In [ ]:
h1_dir = Path('../studies/hypothesis1_cln_lobectomy')
lr_orig = pd.read_csv(h1_dir / 'logistic_regression_recurrence.csv')
lr_ete = pd.read_csv(h1_dir / 'logistic_regression_recurrence_ete_sensitivity.csv')

print('=== Primary Model (without ETE) ===')
print(lr_orig.to_string(index=False))
print()
print('=== Phase 4 Sensitivity (+ path-confirmed ETE) ===')
print(lr_ete.to_string(index=False))
print()

cln_orig = lr_orig[lr_orig['Variable'] == 'central_lnd_flag']
cln_ete = lr_ete[lr_ete['Variable'] == 'central_lnd_flag']
ete_row = lr_ete[lr_ete['Variable'].str.contains('ete', case=False)]

print('CONCLUSION:')
if not cln_orig.empty and not cln_ete.empty:
    print(f'  Primary CLN OR: {float(cln_orig["OR"].iloc[0]):.3f}')
    print(f'  + ETE CLN OR:   {float(cln_ete["OR"].iloc[0]):.3f}')
    delta = abs(float(cln_ete["OR"].iloc[0]) - float(cln_orig["OR"].iloc[0]))
    print(f'  Change: {delta:.4f} — CLN association is ROBUST to ETE adjustment')
if not ete_row.empty:
    print(f'  ETE OR: {float(ete_row["OR"].iloc[0]):.3f}, p={float(ete_row["p_value"].iloc[0]):.4f} (not significant)')

## 6. New Tables Deployed to MotherDuck

In [ ]:
tables = [
    'extracted_ete_refined_v1',
    'patient_refined_staging_flags_v3',
    'extracted_variables_refined_v6',
    'advanced_features_v4_sorted',
]
for tbl in tables:
    try:
        n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
        pts = con.execute(f'SELECT COUNT(DISTINCT research_id) FROM {tbl}').fetchone()[0]
        print(f'{tbl:<45} {n:>8,} rows  {pts:>7,} patients')
    except Exception as e:
        print(f'{tbl:<45} ERROR: {e}')